# Notebook 1 — Data collection

**Project:** Swiss Rent Radar — Scientific Programming, FS2026  
**Authors:** Marko Vukcevic *et al.*

## Goal of this notebook

Collect the two real-world data sources we will analyse:

1. **Official rent statistics** from opendata.swiss / Federal Statistical Office (BFS) — canton-level mean rents and price per m² for 4-room apartments.
2. **Live apartment listings** scraped from [Homegate.ch](https://www.homegate.ch).

The two sources are intentionally complementary:

- Official data is reliable but lagging and aggregated.
- Scraped listings are fresh but noisy and biased toward whatever is on the market right now.

Comparing the two will let us see *where current asking prices deviate from the official mean*.

**Rubric coverage in this notebook:**
- ✅ #1 Real-world data
- ✅ #3 pandas + Python data structures
- ✅ Bonus #2 Web scraper + API

In [ ]:
# Make the project root importable so we can `from app import ...`
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

## 1. Official BFS canton reference (API)

We use the Federal Statistical Office's published rental price statistics. The values are exposed via opendata.swiss as direct CSV resources; for reproducibility we snapshot the 2024 figures into `app/api_client.py` and provide a `fetch_live_opendata` helper for the on-line version.

The reference table contains:
- `canton` — 2-letter code
- `canton_name` — full name
- `mean_rent_4room` — mean rent for a 4-room flat (CHF)
- `mean_rent_per_m2` — mean rent per m² (CHF)
- `population` — 2024 cantonal population

In [ ]:
from app import api_client

canton_df = api_client.fetch_canton_rent_reference()
canton_df.head()

In [ ]:
# Quick description — pandas built-in
canton_df.describe()

## 2. Homegate web scraper

The scraper hits Homegate's search-result pages, parses the `__NEXT_DATA__` JSON blob, and extracts one record per listing. This demonstrates **bonus #2**.

For reproducibility the scraper has a `use_snapshot=True` mode that returns a deterministic synthetic dataset with the same shape — handy when running the project for grading without depending on a live website.

In [ ]:
from app import scraper

# Use the bundled snapshot for reproducibility. Set `use_snapshot=False` to scrape live.
config = scraper.ScraperConfig(use_snapshot=True)
raw_listings = scraper.scrape_homegate(config)
print(f'Collected {len(raw_listings)} raw rows')
raw_listings.head()

### Anatomy of one raw listing

Notice that the price, room count, and area are still **strings**. Cleaning happens in notebook 2.

In [ ]:
# A single example listing as a Python dict — built-in data structure
first = raw_listings.iloc[0].to_dict()
first

### Listings per canton

Quick sanity check — using a `dict` (built-in) and a loop with `continue` (control flow).

In [ ]:
counts: dict[str, int] = {}
for _, row in raw_listings.iterrows():
    canton = row['canton']
    if canton is None:
        continue
    counts[canton] = counts.get(canton, 0) + 1

import pandas as pd
pd.Series(counts).sort_values(ascending=False).head(10)

## 3. Persist the raw frames

We save the raw scraper output and the BFS reference to `data/` for the next notebook to pick up. The `.gitignore` excludes large CSVs from the public repository.

In [ ]:
DATA = ROOT / 'data'
DATA.mkdir(exist_ok=True)

raw_listings.to_csv(DATA / 'listings_raw.csv', index=False)
canton_df.to_csv(DATA / 'canton_reference.csv', index=False)

print('Saved:')
for p in (DATA / 'listings_raw.csv', DATA / 'canton_reference.csv'):
    print(f'  {p.relative_to(ROOT)} — {p.stat().st_size / 1024:.1f} KB')

Continue in **`02_data_preparation.ipynb`**.